## hotpot

In [2]:

import json
from pathlib import Path
import pandas as pd

MAX_CONTEXTS=3000

project_root = Path("D:/Code/jupyter/knowledge_graph")
inputdirectory = project_root / "data_input" / "dataset" / "hotpot" / "hotpot_train_v1.1.json"
outputdirectory = project_root/ "data_input" / "intent_classfication"
with open(inputdirectory, "r", encoding="utf-8") as f:
    hotpot_data = json.load(f)


relations = []
# 使用 question_index 作为 context_id，与分块代码中的逻辑保持一致
question_index = 0 
for item in hotpot_data:
    # 使用 MAX_CONTEXTS 限制处理的 Question 数量
    if question_index >= MAX_CONTEXTS:
        break
    
    question = item["question"].strip()
    # HotpotQA 的 'answer' 是一个字符串，而不是 SQuAD 中嵌入在 QA 列表里的结构
    answer = item["answer"].strip() 
    
    # 🌟 核心：将 Question 索引作为 context_id
    relations.append({
        "question": question,
        "answer": answer,
        "context_id": question_index,      # 整数索引，与 chunk.csv 关联
    })
    
    question_index += 1

df_qa = pd.DataFrame(relations)

df_graph = df_qa.copy()

# 保存 QA 文件
df_graph.to_csv(outputdirectory/"hotpot.csv", sep="|", index=False)
print("✅ qa.csv 生成完毕:", df_graph.shape)

# 打印示例数据
print("\nQA Data Sample:")
print(df_graph.head())

✅ qa.csv 生成完毕: (3000, 3)

QA Data Sample:
                                            question                   answer  \
0  Which magazine was started first Arthur's Maga...        Arthur's Magazine   
1  The Oberoi family is part of a hotel company t...                    Delhi   
2  Musician and satirist Allie Goertz wrote a son...  President Richard Nixon   
3    What nationality was James Henry Miller's wife?                 American   
4  Cadmium Chloride is slightly soluble in this c...                  alcohol   

   context_id  
0           0  
1           1  
2           2  
3           3  
4           4  


## WebQSP

In [9]:

# 设定最大处理的问题数量，与您的 HotpotQA 脚本保持一致
MAX_CONTEXTS = 4500

# --- 路径配置 (请根据您的实际文件位置修改) ---
project_root = Path("D:/Code/jupyter/knowledge_graph")
# WebQSP 训练数据路径
inputdirectory_webqsp = project_root / "data_input" / "dataset" / "webqa" / "WebQSP.train.json"
# 输出目录
outputdirectory = project_root / "data_input" / "intent_classfication"

# 确保输出目录存在
outputdirectory.mkdir(parents=True, exist_ok=True)
# --------------------------------------------------

print(f"📖 正在加载 WebQSP 数据集: {inputdirectory_webqsp.name}")
try:
    with open(inputdirectory_webqsp, "r", encoding="utf-8") as f:
        webqsp_data = json.load(f)
except FileNotFoundError:
    print(f"❌ 错误: 找不到文件 {inputdirectory_webqsp}")
    exit()

# WebQSP 的数据结构是包含一个 'Questions' 键的字典
questions_list = webqsp_data.get("Questions", [])

relations = []
# 使用 question_index 作为 context_id
question_index = 3000

print(f"⏳ 开始处理前 {MAX_CONTEXTS} 个问题...")

for item in questions_list:
    # 使用 MAX_CONTEXTS 限制处理的 Question 数量
    if question_index >= MAX_CONTEXTS:
        break
    
    # 原始问题
    question = item.get("RawQuestion", "").strip()
    
    # 核心：提取答案
    # WebQSP 的一个问题可能有多个 'Parses'，每个 'Parse' 可能有多个 'Answers'。
    # 我们将收集所有解析中的所有答案。
    
    all_answers = []
    
    # 遍历所有的解析 (Parses)
    for parse in item.get("Parses", []):
        # 遍历该解析下的所有答案
        for answer_obj in parse.get("Answers", []):
            # WebQSP 的答案是 "EntityName" 或 "AnswerArgument" (对于 Literal/Value answers)
            answer_text = answer_obj.get("EntityName") or answer_obj.get("AnswerArgument")
            if answer_text:
                all_answers.append(str(answer_text).strip())
    
    # 答案去重并合并成一个字符串（如果一个问题有多个答案，用分号或逗号分隔，这里使用逗号分隔）
    # 注意：在实际 RAG/KG 应用中，保留多个答案可能更合适。这里为了与您的 HotpotQA 脚本结构保持一致，将其合并。
    unique_answers = sorted(list(set(all_answers)))
    final_answer = ", ".join(unique_answers)
    # 限制答案长度不超过50个字符
    if len(final_answer) > 50:
        # 截断并添加省略号（注意：省略号也算字符）
        final_answer = final_answer[:47] + "..."

    # 仅当问题和答案都存在时才添加
    if question and final_answer:
        relations.append({
            "question": question,
            "answer": final_answer,
            "context_id": question_index, # 整数索引，用于关联
        })
        
        question_index += 1

# 创建 DataFrame
df_qa_webqsp = pd.DataFrame(relations)
df_graph_webqsp = df_qa_webqsp.copy()

# 保存 QA 文件
output_path = outputdirectory / "webqsp_qa.csv"
df_graph_webqsp.to_csv(output_path, sep="|", index=False)
print(f"\n✅ webqsp_qa.csv 生成完毕: {df_graph_webqsp.shape}")

# 打印示例数据
print("\nWebQSP QA Data Sample:")
print(df_graph_webqsp.head())

📖 正在加载 WebQSP 数据集: WebQSP.train.json
⏳ 开始处理前 4500 个问题...

✅ webqsp_qa.csv 生成完毕: (1500, 3)

WebQSP QA Data Sample:
                                            question           answer  \
0         what is the name of justin bieber brother?     Jaxon Bieber   
1  what character did natalie portman play in sta...    Padmé Amidala   
2        what country is the grand bahama island in?          Bahamas   
3             what kind of money to take to bahamas?  Bahamian dollar   
4  what character did john noble play in lord of ...      Denethor II   

   context_id  
0        3000  
1        3001  
2        3002  
3        3003  
4        3004  
